# 台灣信用卡違約預警：可行性測試

這份筆記本用來產出期中研究計畫需要的**實際執行數字**，所有結果都可由本筆記本重現（隨機種子固定為 42）。

| 產出 | 用在哪裡 |
|---|---|
| 資料集描述（筆數、違約比例、欄位、資料問題） | 簡報第 18 頁、Word 表 3-1 |
| 前處理決策與理由 | 簡報第 19 頁、Word 參章三 |
| Baseline 模型比較（含交叉驗證、訓練時間、混淆矩陣） | 簡報第 23 頁、Word 表 4-1、圖 4-1 |
| RQ1 公平性：移除敏感屬性、各族群誤判率 | 簡報第 23 頁、Word 肆章二 |
| RQ2 可解釋性：邏輯迴歸加特徵設計 vs XGBoost | 簡報第 23 頁、Word 肆章二 |
| RQ3 預警時機與金額：提早預警曲線、名單涵蓋金額 | 簡報第 23 頁、Word 肆章二 |

**使用方式（Google Colab）**：上方選單「執行階段」→「全部執行」。程式會自動從 UCI 下載資料、安裝中文字型。全部跑完約需幾分鐘，圖表存在 `figures/`、數字表格存在 `results/`，最後一格會打包成 zip 下載。

**注意**：這是期中的可行性驗證，目的是證明研究做得出來。期末的正式實驗會再調整參數、門檻，並做完整的交叉驗證與錯誤分析。

## 0. 環境設定

In [ ]:
# 安裝 Colab 預設沒有的套件
import subprocess, sys, importlib.util
for pkg in ["xgboost", "xlrd"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, time, shutil, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

RANDOM_STATE = 42
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
pd.set_option("display.max_columns", 40)

# 中文字型：Colab 沒有內建中文字型，先安裝 Noto CJK；本機 Windows 則改用微軟正黑體
FONT_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if not os.path.exists(FONT_PATH):
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        pass  # 非 Linux 環境（例如 Windows）沒有 apt-get
if os.path.exists(FONT_PATH):
    font_manager.fontManager.addfont(FONT_PATH)
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=FONT_PATH).get_name()
else:
    plt.rcParams["font.family"] = ["Microsoft JhengHei", "PingFang TC", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
print("環境設定完成，字型：", plt.rcParams["font.family"])

## 1. 載入資料

資料來源：UCI Machine Learning Repository「Default of Credit Card Clients」，授權 CC BY 4.0。
原始論文：Yeh, I.-C., & Lien, C.-H. (2009). *Expert Systems with Applications*, 36(2), 2473–2480.

程式會優先讀取同資料夾的檔案；沒有的話自動從 UCI 下載。

In [ ]:
UCI_ZIP = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"

def load_raw():
    # 1) 同資料夾已有 CSV 或 XLS 就直接讀
    if os.path.exists("default_of_credit_card_clients.csv"):
        return pd.read_csv("default_of_credit_card_clients.csv")
    if os.path.exists("default of credit card clients.xls"):
        return pd.read_excel("default of credit card clients.xls", header=1)
    # 2) 否則從 UCI 下載 zip 並解壓縮
    urllib.request.urlretrieve(UCI_ZIP, "credit.zip")
    with zipfile.ZipFile("credit.zip") as z:
        z.extractall("credit_raw")
    for root, _, files in os.walk("credit_raw"):
        for f in files:
            if f.endswith(".xls"):
                # XLS 第一列是 X1~X23 代號，第二列才是欄位名稱，所以 header=1
                return pd.read_excel(os.path.join(root, f), header=1)
    raise FileNotFoundError("找不到資料檔，請手動下載後放在同資料夾")

raw = load_raw()
# 欄位改名：PAY_0 改成 PAY_1，讓 1~6 都代表「距今第幾個月」（1 = 2005 年 9 月，6 = 2005 年 4 月）
df = raw.rename(columns={"PAY_0": "PAY_1", "default payment next month": "default"}).drop(columns=["ID"])
print("資料維度：", df.shape)
df.head()

## 2. 資料集描述（簡報第 18 頁、Word 表 3-1）

In [ ]:
n = len(df)
n_default = int(df["default"].sum())
dup = int(df.duplicated().sum())
desc = pd.DataFrame({
    "項目": ["資料來源與授權", "資料期間", "資料型態", "樣本數", "特徵數", "類別分布", "缺失值", "完全重複的列（不含 ID）"],
    "內容": ["UCI Default of Credit Card Clients，CC BY 4.0（Yeh & Lien, 2009）",
             "台灣某銀行持卡人 2005 年 4 月至 9 月之帳務紀錄，預測 10 月是否違約",
             "結構化表格",
             f"{n:,} 位持卡人",
             f"{df.shape[1]-1} 個（額度、人口屬性 4 個、近 6 個月繳款狀態／帳單金額／還款金額）",
             f"違約 {n_default:,} 人（{n_default/n:.1%}）、未違約 {n-n_default:,} 人（{1-n_default/n:.1%}）",
             f"{int(df.isnull().sum().sum())} 筆",
             f"{dup} 列"],
})
desc.to_csv("results/dataset_description.csv", index=False, encoding="utf-8-sig")
desc

### 2.1 說明文件未定義的代碼

UCI 說明文件定義：教育程度 1 = 研究所、2 = 大學、3 = 高中、4 = 其他；婚姻 1 = 已婚、2 = 單身、3 = 其他；
繳款狀態 -1 = 準時繳清、1~9 = 延遲 1~9 個月。但實際資料還出現下列未定義的代碼，需在前處理時決定怎麼處理。

In [ ]:
pay_cols = [f"PAY_{i}" for i in range(1, 7)]
undoc = pd.DataFrame({
    "欄位": ["EDUCATION", "EDUCATION", "EDUCATION", "MARRIAGE", "PAY_1（最近一個月）", "PAY_1（最近一個月）"],
    "未定義代碼": [0, 5, 6, 0, -2, 0],
    "筆數": [int((df.EDUCATION == 0).sum()), int((df.EDUCATION == 5).sum()), int((df.EDUCATION == 6).sum()),
            int((df.MARRIAGE == 0).sum()), int((df.PAY_1 == -2).sum()), int((df.PAY_1 == 0).sum())],
})
undoc.to_csv("results/undocumented_codes.csv", index=False, encoding="utf-8-sig")
undoc

### 2.2 探索性分析圖表

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 圖 A：類別分布
counts = df["default"].value_counts().sort_index()
axes[0].bar(["未違約 (0)", "違約 (1)"], counts.values, color=["#4C72B0", "#C44E52"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}\n({v/n:.1%})", ha="center", va="bottom")
axes[0].set_title("類別分布")
axes[0].set_ylabel("人數")
axes[0].set_ylim(0, counts.max() * 1.2)

# 圖 B：最近一個月繳款狀態與違約率
g = df.groupby("PAY_1")["default"].agg(["mean", "count"])
axes[1].bar(g.index.astype(str), g["mean"], color="#55A868")
axes[1].set_title("最近一個月繳款狀態與下月違約率")
axes[1].set_xlabel("PAY_1（-2、-1、0：未延遲類；1 以上：延遲月數）")
axes[1].set_ylabel("違約率")
plt.tight_layout()
plt.savefig("figures/eda_class_and_pay1.png", dpi=200)
plt.show()

print("各繳款狀態的人數與違約率：")
print(g.assign(mean=lambda d: d["mean"].round(3)).rename(columns={"mean": "違約率", "count": "人數"}).T.to_string())

In [ ]:
# 各人口屬性族群的違約率（公平性分析的背景資訊）
age_bins = [0, 29, 39, 49, 200]
age_labels = ["29 歲以下", "30-39 歲", "40-49 歲", "50 歲以上"]
df["AGE_GROUP"] = pd.cut(df["AGE"], bins=age_bins, labels=age_labels)
sex_map = {1: "男性", 2: "女性"}

edu_map = {1: "研究所", 2: "大學", 3: "高中", 4: "其他", 0: "未定義(0)", 5: "未定義(5)", 6: "未定義(6)"}
mar_map = {1: "已婚", 2: "單身", 3: "其他", 0: "未定義(0)"}

rows = []
for col, mapping in [("SEX", sex_map), ("AGE_GROUP", None), ("EDUCATION", edu_map), ("MARRIAGE", mar_map)]:
    for k, sub in df.groupby(col, observed=True):
        rows.append([col, mapping.get(k, k) if mapping else k, len(sub), round(sub["default"].mean(), 3)])
group_rate = pd.DataFrame(rows, columns=["屬性", "族群", "人數", "違約率"])
group_rate.to_csv("results/group_default_rate.csv", index=False, encoding="utf-8-sig")
group_rate

## 3. 資料前處理（簡報第 19 頁、Word 參章三）

| 步驟 | 做法 | 理由 |
|---|---|---|
| 未定義代碼 | 教育程度 0、5、6 併入 4（其他）；婚姻 0 併入 3（其他） | 說明文件無定義，筆數少，併入「其他」可保留樣本又不自行猜測意義 |
| 繳款狀態 -2、0 | 保留原值，當作有順序的數值 | 數值越大代表延遲越久，順序資訊有意義；期末再比較改用類別編碼的效果 |
| 類別編碼 | 教育程度、婚姻做 One-Hot 編碼 | 這兩個欄位沒有大小順序 |
| 標準化 | 數值欄位 StandardScaler | 邏輯迴歸對尺度敏感 |
| 資料切分 | 80% 訓練、20% 測試，分層抽樣，random_state = 42 | 保持兩邊違約比例一致 |
| 類別不平衡 | 三個模型都使用類別權重 | 違約只占約 22%，不加權重會傾向全部預測「不違約」 |
| 防止資料洩漏 | 編碼與標準化都包在 Pipeline 裡，只用訓練資料 fit | 測試集資訊不會影響前處理參數 |

In [ ]:
def clean(d):
    d = d.copy()
    d["EDUCATION"] = d["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
    d["MARRIAGE"] = d["MARRIAGE"].replace({0: 3})
    return d

data = clean(df)
CAT_COLS = ["EDUCATION", "MARRIAGE"]
SENSITIVE = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]

def feature_cols(lag=0, sensitive=True):
    # lag = 0 使用全部 6 個月；lag = k 代表拿掉最近 k 個月，模擬提早 k+1 個月預警
    months = range(1 + lag, 7)
    cols = (["LIMIT_BAL"] + [f"PAY_{m}" for m in months]
            + [f"BILL_AMT{m}" for m in months] + [f"PAY_AMT{m}" for m in months])
    return cols + (SENSITIVE if sensitive else [])

train, test = train_test_split(data, test_size=0.2, random_state=RANDOM_STATE, stratify=data["default"])
y_train, y_test = train["default"], test["default"]
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"訓練集 {len(train):,} 筆（違約 {y_train.mean():.1%}），測試集 {len(test):,} 筆（違約 {y_test.mean():.1%}）")

def make_model(name, cols):
    cat = [c for c in CAT_COLS if c in cols]
    num = [c for c in cols if c not in cat]
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                             ("num", StandardScaler(), num)])
    if name == "邏輯迴歸":
        clf = LogisticRegression(max_iter=3000, class_weight="balanced")
    elif name == "決策樹 (深度 4)":
        clf = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=RANDOM_STATE)
    else:
        clf = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, scale_pos_weight=pos_weight,
                            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    return Pipeline([("pre", pre), ("clf", clf)])

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 4. Baseline 模型比較（簡報第 23 頁、Word 表 4-1）

- 測試集指標以門檻 0.5 計算，Precision、Recall、F1 皆以「違約」為正類。
- 另以訓練集做 5 折交叉驗證，AUC 以「平均值 ± 標準差」呈現（Word 表 4-1 的註要求）。

In [ ]:
MODEL_NAMES = ["邏輯迴歸", "決策樹 (深度 4)", "XGBoost"]
cols_all = feature_cols()
rows, fitted, probs = [], {}, {}
for name in MODEL_NAMES:
    m = make_model(name, cols_all)
    t0 = time.time()
    m.fit(train[cols_all], y_train)
    train_time = time.time() - t0
    p = m.predict_proba(test[cols_all])[:, 1]
    pred = (p >= 0.5).astype(int)
    cv_auc = cross_val_score(make_model(name, cols_all), train[cols_all], y_train, cv=CV, scoring="roc_auc")
    rows.append([name, accuracy_score(y_test, pred), precision_score(y_test, pred), recall_score(y_test, pred),
                 f1_score(y_test, pred), roc_auc_score(y_test, p), f"{cv_auc.mean():.3f} ± {cv_auc.std():.3f}",
                 round(train_time, 2)])
    fitted[name], probs[name] = m, p

baseline = pd.DataFrame(rows, columns=["模型", "Accuracy", "Precision", "Recall", "F1-score", "測試集 AUC",
                                       "5 折 CV AUC", "訓練時間（秒）"]).round(3)
baseline.to_csv("results/table4_1_baseline.csv", index=False, encoding="utf-8-sig")
baseline

In [ ]:
# 混淆矩陣（Word 圖 4-1）
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, name in zip(axes, MODEL_NAMES):
    cm = confusion_matrix(y_test, (probs[name] >= 0.5).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=["未違約", "違約"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name)
    ax.set_xlabel("預測標籤")
    ax.set_ylabel("真實標籤")
plt.tight_layout()
plt.savefig("figures/fig4_1_confusion_matrices.png", dpi=200)
plt.show()

# 參考：只用「最近一個月繳款狀態」一條規則能做到多好（用來說明單一特徵的主導性）
rule = DecisionTreeClassifier(max_depth=1, class_weight="balanced", random_state=RANDOM_STATE).fit(train[["PAY_1"]], y_train)
rp = rule.predict(test[["PAY_1"]])
print(f"只用 PAY_1 切一刀：Recall = {recall_score(y_test, rp):.3f}，F1 = {f1_score(y_test, rp):.3f}")

## 5. RQ1 公平性：移除敏感屬性的影響

依金管會《金融業運用人工智慧（AI）之指引》的公平性與資料最小化原則，比較「使用」與「不使用」性別、教育、婚姻、年齡四個敏感屬性時：
1. 整體預測力（AUC）差多少；
2. 不同族群（性別、年齡層、性別 × 年齡層）的漏判率（FNR：實際違約卻判為不違約）與誤判率（FPR：實際未違約卻判為違約）。

模型使用 XGBoost，門檻 0.5。族群分析一律用原始屬性分組，即使模型沒有使用這些欄位。

In [ ]:
def group_errors(y_true, pred, groups):
    out = []
    for g, idx in groups.groupby(groups, observed=True).groups.items():
        yt, yp = y_true.loc[idx], pd.Series(pred, index=y_true.index).loc[idx]
        pos, neg = (yt == 1).sum(), (yt == 0).sum()
        out.append([g, len(yt), round(yt.mean(), 3),
                    round(((yp == 0) & (yt == 1)).sum() / pos, 3) if pos else np.nan,
                    round(((yp == 1) & (yt == 0)).sum() / neg, 3) if neg else np.nan])
    return pd.DataFrame(out, columns=["族群", "人數", "實際違約率", "FNR 漏判率", "FPR 誤判率"])

test_groups = {
    "性別": test["SEX"].map(sex_map),
    "年齡層": test["AGE_GROUP"].astype(str),
    "性別 × 年齡層": test["SEX"].map(sex_map) + "・" + test["AGE_GROUP"].astype(str),
}

fair_rows, fair_tables = [], []
for use_s in [True, False]:
    cols = feature_cols(sensitive=use_s)
    m = make_model("XGBoost", cols).fit(train[cols], y_train)
    p = m.predict_proba(test[cols])[:, 1]
    pred = (p >= 0.5).astype(int)
    cv_auc = cross_val_score(make_model("XGBoost", cols), train[cols], y_train, cv=CV, scoring="roc_auc")
    label = "使用敏感屬性" if use_s else "移除敏感屬性"
    fair_rows.append([label, round(roc_auc_score(y_test, p), 3), f"{cv_auc.mean():.3f} ± {cv_auc.std():.3f}",
                      round(f1_score(y_test, pred), 3)])
    for gname, gser in test_groups.items():
        t = group_errors(y_test, pred, gser)
        t.insert(0, "分組方式", gname)
        t.insert(0, "模型", label)
        fair_tables.append(t)

fair_summary = pd.DataFrame(fair_rows, columns=["模型", "測試集 AUC", "5 折 CV AUC", "F1-score"])
fair_detail = pd.concat(fair_tables, ignore_index=True)
fair_summary.to_csv("results/rq1_fairness_summary.csv", index=False, encoding="utf-8-sig")
fair_detail.to_csv("results/rq1_fairness_by_group.csv", index=False, encoding="utf-8-sig")
display(fair_summary)
fair_detail

In [ ]:
# 各族群漏判率比較圖（移除敏感屬性 vs 使用敏感屬性）
sub = fair_detail[fair_detail["分組方式"] == "性別 × 年齡層"].pivot(index="族群", columns="模型", values="FNR 漏判率")
ax = sub[["使用敏感屬性", "移除敏感屬性"]].plot(kind="bar", figsize=(11, 4.5), color=["#8172B2", "#64B5CD"])
ax.set_title("各族群漏判率（FNR）：使用 vs 移除敏感屬性")
ax.set_ylabel("漏判率（實際違約卻判為不違約）")
ax.set_xlabel("")
ax.set_ylim(0, sub.values.max() * 1.25)
ax.legend(title="", loc="upper center", ncol=2)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("figures/rq1_fnr_by_group.png", dpi=200)
plt.show()

## 6. RQ2 可解釋性：簡單模型加特徵設計，能追上 XGBoost 嗎？

金融業要能說明判斷理由，邏輯迴歸的係數容易解釋，XGBoost 較難。這裡在邏輯迴歸加入人看得懂的衍生特徵，比較和 XGBoost 的差距：

- 額度使用率：每月帳單金額 ÷ 信用額度
- 還款比例：當月還款金額 ÷ 上月帳單金額
- 近 6 個月延遲次數、最長延遲月數
- 使用率變化：最近一個月使用率 − 六個月前使用率

衍生特徵都是單筆資料內的計算，不用到其他人的資料，不會造成資料洩漏。

In [ ]:
def add_features(d):
    d = d.copy()
    for m in range(1, 7):
        d[f"UTIL_{m}"] = d[f"BILL_AMT{m}"] / d["LIMIT_BAL"]
    for m in range(1, 6):
        d[f"PAYRATIO_{m}"] = d[f"PAY_AMT{m}"] / d[f"BILL_AMT{m+1}"].clip(lower=1)
    d["N_DELAY"] = (d[pay_cols] > 0).sum(axis=1)
    d["MAX_DELAY"] = d[pay_cols].max(axis=1)
    d["UTIL_TREND"] = d["UTIL_1"] - d["UTIL_6"]
    return d

train_fe, test_fe = add_features(train), add_features(test)
eng_cols = cols_all + [c for c in train_fe.columns if c.startswith(("UTIL_", "PAYRATIO_", "N_DELAY", "MAX_DELAY"))]

rows = []
for label, name, cols, tr_, te_ in [("邏輯迴歸（原始特徵）", "邏輯迴歸", cols_all, train, test),
                                    ("邏輯迴歸（加衍生特徵）", "邏輯迴歸", eng_cols, train_fe, test_fe),
                                    ("XGBoost（原始特徵）", "XGBoost", cols_all, train, test)]:
    m = make_model(name, cols).fit(tr_[cols], y_train)
    p = m.predict_proba(te_[cols])[:, 1]
    cv_auc = cross_val_score(make_model(name, cols), tr_[cols], y_train, cv=CV, scoring="roc_auc")
    rows.append([label, len(cols), round(roc_auc_score(y_test, p), 3), f"{cv_auc.mean():.3f} ± {cv_auc.std():.3f}",
                 round(f1_score(y_test, (p >= 0.5).astype(int)), 3)])
rq2 = pd.DataFrame(rows, columns=["模型", "特徵數", "測試集 AUC", "5 折 CV AUC", "F1-score"])
rq2.to_csv("results/rq2_interpretability.csv", index=False, encoding="utf-8-sig")
rq2

## 7. RQ3 預警時機：越早預警，準確度掉多少？

拿掉最近 k 個月的資料，模擬「提早 k+1 個月」預警。例如提早 2 個月，代表只用 2005 年 4 到 8 月的資料，預測 10 月是否違約。

In [ ]:
months_label = ["9 月", "8 月", "7 月", "6 月", "5 月"]
rows = []
for lag in range(0, 5):
    cols = feature_cols(lag=lag)
    m = make_model("XGBoost", cols).fit(train[cols], y_train)
    p = m.predict_proba(test[cols])[:, 1]
    cv_auc = cross_val_score(make_model("XGBoost", cols), train[cols], y_train, cv=CV, scoring="roc_auc")
    rows.append([lag + 1, f"4 月 ~ {months_label[lag]}", round(roc_auc_score(y_test, p), 3),
                 round(cv_auc.mean(), 3), round(cv_auc.std(), 3)])
rq3_time = pd.DataFrame(rows, columns=["提早幾個月預警", "使用資料期間", "測試集 AUC", "CV AUC 平均", "CV AUC 標準差"])
rq3_time.to_csv("results/rq3_lead_time.csv", index=False, encoding="utf-8-sig")
display(rq3_time)

plt.figure(figsize=(7.5, 4.3))
plt.errorbar(rq3_time["提早幾個月預警"], rq3_time["CV AUC 平均"], yerr=rq3_time["CV AUC 標準差"],
             marker="o", capsize=4, color="#C44E52")
plt.title("預警時間與預測力（XGBoost，5 折交叉驗證）")
plt.xlabel("提早幾個月預警")
plt.ylabel("AUC")
plt.xticks(rq3_time["提早幾個月預警"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/rq3_lead_time.png", dpi=200)
plt.show()

## 8. RQ3 預警金額：聯絡前幾成客戶，能涵蓋多少違約金額？

依 XGBoost 的違約機率由高到低排序，計算聯絡前 k% 客戶時，涵蓋了多少比例的違約人數，以及違約者的帳單金額（9 月帳單，負值以 0 計）。
隨機抽樣聯絡 k% 客戶，期望只能涵蓋 k%，作為比較基準。

In [ ]:
p = probs["XGBoost"]
order = np.argsort(-p)
is_bad = y_test.values == 1
exposure = test["BILL_AMT1"].clip(lower=0).values
rows = []
for k in [5, 10, 15, 20, 30, 40, 50]:
    top = order[: int(len(order) * k / 100)]
    rows.append([k, round(is_bad[top].sum() / is_bad.sum(), 3),
                 round(exposure[top][is_bad[top]].sum() / exposure[is_bad].sum(), 3)])
rq3_money = pd.DataFrame(rows, columns=["聯絡前 k% 客戶", "涵蓋違約人數比例", "涵蓋違約帳單金額比例"])
rq3_money.to_csv("results/rq3_money_capture.csv", index=False, encoding="utf-8-sig")
display(rq3_money)

plt.figure(figsize=(7.5, 4.3))
plt.plot(rq3_money["聯絡前 k% 客戶"], rq3_money["涵蓋違約帳單金額比例"] * 100, marker="o", label="依模型排序：違約金額")
plt.plot(rq3_money["聯絡前 k% 客戶"], rq3_money["涵蓋違約人數比例"] * 100, marker="s", label="依模型排序：違約人數")
plt.plot([0, 50], [0, 50], linestyle="--", color="gray", label="隨機聯絡")
plt.title("風險名單涵蓋率（XGBoost）")
plt.xlabel("聯絡前 k% 高風險客戶")
plt.ylabel("涵蓋比例（%）")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/rq3_money_capture.png", dpi=200)
plt.show()

## 9. 重點數字整理（直接給簡報、Word 使用）

In [ ]:
xgb_row = baseline.set_index("模型").loc["XGBoost"]
lr_row = baseline.set_index("模型").loc["邏輯迴歸"]
fs = fair_summary.set_index("模型")
top20 = rq3_money.set_index("聯絡前 k% 客戶").loc[20]
summary_lines = [
    f"資料：{n:,} 位持卡人，違約率 {n_default/n:.1%}",
    f"Baseline：邏輯迴歸 AUC {lr_row['測試集 AUC']}、F1 {lr_row['F1-score']}；XGBoost AUC {xgb_row['測試集 AUC']}、F1 {xgb_row['F1-score']}",
    f"RQ1：移除敏感屬性後 AUC 由 {fs.loc['使用敏感屬性','測試集 AUC']} 變為 {fs.loc['移除敏感屬性','測試集 AUC']}",
    f"RQ2：邏輯迴歸 AUC 由 {rq2.iloc[0]['測試集 AUC']}（原始）提升到 {rq2.iloc[1]['測試集 AUC']}（加衍生特徵），XGBoost 為 {rq2.iloc[2]['測試集 AUC']}",
    f"RQ3：提早 1 個月 CV AUC {rq3_time.iloc[0]['CV AUC 平均']}，提早 2 個月 {rq3_time.iloc[1]['CV AUC 平均']}，提早 5 個月 {rq3_time.iloc[4]['CV AUC 平均']}",
    f"RQ3：聯絡前 20% 高風險客戶，涵蓋 {top20['涵蓋違約人數比例']:.1%} 的違約者、{top20['涵蓋違約帳單金額比例']:.1%} 的違約帳單金額",
]
with open("results/summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))
print("\n".join(summary_lines))
print("\n圖表：", sorted(os.listdir("figures")))
print("表格：", sorted(os.listdir("results")))

In [ ]:
# 打包圖表與表格（在 Colab 會自動下載 zip）
with zipfile.ZipFile("feasibility_outputs.zip", "w") as z:
    for folder in ["figures", "results"]:
        for f in os.listdir(folder):
            z.write(os.path.join(folder, f))
try:
    from google.colab import files
    files.download("feasibility_outputs.zip")
except ImportError:
    print("已建立 feasibility_outputs.zip（非 Colab 環境，請自行到資料夾取用）")

## 10. 解讀提醒與限制（寫報告時請注意）

- 這是**單次切分加 5 折交叉驗證**的可行性結果，期末正式實驗應再調整超參數與門檻，並做錯誤分析。
- 資料為 2005 年台灣卡債風暴時期的資料，結論不一定適用於現在，研究限制需說明。
- 金額涵蓋分析用 9 月帳單金額近似「違約可能造成的損失」，並非實際損失金額。
- 族群誤判率中，部分交叉族群人數較少，差異需謹慎解讀。
- 報告中引用的所有數字，請以你們自己執行本筆記本的結果為準。